# TDNet weekly manual Top-25 ballot

Drag the logo cards into your order, then submit. The editor shows the top 50 candidate teams so the owner has room to reorder, but only the first 25 submitted teams receive poll points. Raw ballots and tables are saved under `data/publication/<season>/manual_polls/`; curated figures are saved under `publication/<season>/figures/`. The poll uses the enabled margin-objective members in the weekly learned-model inventory, including KNN when enabled, while excluding explicit naive baselines. The owner ballot is separate.

In [ ]:
from pathlib import Path
import json
import os
import sys
import tempfile
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / 'src' / 'gridiron_ml').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Could not find the TDNet repo root from the current notebook directory.')
MPLCONFIGDIR = Path(tempfile.gettempdir()) / 'tdnet_matplotlib_cache'
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from gridiron_ml.publication import (
    load_saved_ballot, make_drag_poll_editor,
    run_manual_poll, validate_ballot,
)
from gridiron_ml.publication.poll_recaps import plot_tdnet_vs_ap_poll
from gridiron_ml.publication.polls import load_ap_top25
from gridiron_ml.publication.roster_poll import build_frozen_roster_poll


In [ ]:
# Change these two values every week. WEEK is the poll snapshot week, not the upcoming game week.
SEASON = 2026
WEEK = 0
TOP_N = 25
DISPLAY_N = 50
BALLOT_NAME = 'my_manual_poll'
LOGO_DIR = PROJECT_ROOT / 'data' / 'meta' / 'logos' / 'by_team'
MODELS_ROOT = PROJECT_ROOT / 'models'
WEEKLY_INVENTORY = PROJECT_ROOT / 'docs' / 'publication_2026' / 'weekly_learned_model_inventory.csv'
AP_RANKINGS = PROJECT_ROOT / 'data' / 'raw' / 'cfbd' / 'v2' / 'rankings' / f'{SEASON}.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'publication' / str(SEASON) / 'manual_polls' / f'week_{WEEK:02d}'
FIGURE_DIR = PROJECT_ROOT / 'publication' / str(SEASON) / 'figures' / 'manual_polls' / f'week_{WEEK:02d}'

seed = build_frozen_roster_poll(
    WEEKLY_INVENTORY,
    season=SEASON,
    week=WEEK,
    output_dir=OUTPUT_DIR / 'model_seed',
    project_root=PROJECT_ROOT,
    logo_dir=LOGO_DIR,
    top_n=DISPLAY_N,
    objective='margin',
)
seed_teams = seed['poll'].sort_values('rank')['keys_team'].astype(str).head(DISPLAY_N).tolist()
saved = load_saved_ballot(PROJECT_ROOT, season=SEASON, week=WEEK, ballot_name=BALLOT_NAME, top_n=TOP_N)
if saved:
    STARTING_TEAMS = saved + [team for team in seed_teams if team not in set(saved)]
    STARTING_TEAMS = STARTING_TEAMS[:DISPLAY_N]
    SOURCE_NOTE = 'your saved Top 25, extended with learned roster candidates'
else:
    STARTING_TEAMS = seed_teams
    SOURCE_NOTE = 'learned weekly roster candidate poll'

print(f'Initial order: {SOURCE_NOTE}')
print(f'{SEASON} week {WEEK}: showing {len(STARTING_TEAMS)} candidates; scoring only top {TOP_N}')
if len(STARTING_TEAMS) != DISPLAY_N:
    raise ValueError(f'Expected {DISPLAY_N} display candidates; got {len(STARTING_TEAMS)}.')
if len(set(STARTING_TEAMS)) != len(STARTING_TEAMS):
    raise ValueError('Display candidate list contains duplicate teams.')
validate_ballot(STARTING_TEAMS[:TOP_N], top_n=TOP_N)


In [ ]:
state, submit_button, submit_output = make_drag_poll_editor(
    STARTING_TEAMS, logo_dir=LOGO_DIR, top_n=DISPLAY_N, ballot_name=BALLOT_NAME
)

def submit_my_ballot(_):
    with submit_output:
        submit_output.clear_output()
        ordered_candidates = json.loads(state.value)
        teams = ordered_candidates[:TOP_N]
        validate_ballot(teams, top_n=TOP_N)
        print(f'Running the poll with the first {TOP_N} teams from the {DISPLAY_N}-team editor...')
        result = run_manual_poll(
            PROJECT_ROOT, season=SEASON, week=WEEK, teams=teams,
            ballot_name=BALLOT_NAME, top_n=TOP_N, models_root=MODELS_ROOT,
            inventory_path=WEEKLY_INVENTORY, objective='margin',
            logo_dir=LOGO_DIR, output_dir=OUTPUT_DIR, figure_output_dir=FIGURE_DIR,
        )
        display(result['poll'])
        display(Image(filename=str(result['figures']['top25'])))
        display(Image(filename=str(result['figures']['all_ballots'])))
        ap = load_ap_top25(AP_RANKINGS, season=SEASON, week=WEEK) if AP_RANKINGS.exists() else None
        if ap is not None and not ap.empty:
            ap_figure = plot_tdnet_vs_ap_poll(result['poll'].rename(columns={'keys_team': 'team'})[['team', 'rank']], ap[['team', 'rank']], FIGURE_DIR / 'tdnet_vs_ap_top25.png', title=f'{SEASON} Week {WEEK}: TDNet Top 25 vs AP Top 25', logo_dir=LOGO_DIR)
            display(Image(filename=str(ap_figure)))
        else:
            print(f'AP comparison unavailable: no {SEASON} Week {WEEK} AP snapshot is present.')
        print(f'Saved to {result["output_dir"]}')
        print(f'KNN participated: {result["metadata"]["knn_participated"]}')

submit_button.on_click(submit_my_ballot)
print(f'Drag the logo cards, then click the blue submit button above. Only ranks 1-{TOP_N} will score points.')


## Weekly operating note

The saved ballot is keyed by season, week, and ballot name. The next run starts from that saved Top 25, then extends the draggable list with the learned roster's remaining candidates up to 50 teams. The poll output contains `manual_poll_ballots.csv`, `manual_poll_top25.csv`, and a matrix figure with the manual ballot labeled. Only ranks 1-25 from the manual order are persisted and scored.